# ELISa: changing-physics observer batches on your CPU and GPU

This handoff measures the actual speedup on the allocated runtime. **No10×result
is assumed.** Ada geometry-stage gains are not a Colab complete-analysis result.
The same changing q, potential, temperatures, inclination, third light and noise
parameters run through complete reflected curves and the original ELISa
observation mapping and density, with identical host output contracts.

**Scientific scope:** circular detached unspotted/nonpulsating binaries, bounded
inner Roche branches, two finite curvature field models, linear LD, blackbody
radiance and reflection. CPU topology/clipping, exact bisection tails, LD bridges
and transfers remain included. Eccentric/contact/spotted systems are outside this
opt-in path; choose original full-CPU modeling for them. No cached physical curve
or nuisance-only workload is timed as a fresh physical proposal.

**Accuracy:** implementation parity, finite numerical convergence and physical
model adequacy are separate. The demanding1e-4 tests and original nominal
D<=.1 / accumulated log-likelihood enclosure<=.01 guards are retained. Current
field/boundary refinement can fail those guards. Failure is reported and stops
any scientific acceptance; this notebook does not run a fit, qualified MCMC,
posterior convergence or ESS experiment. It is a reproducible finite-model
validation/cost handoff, not a qualified TESS analysis.

**Execution status:** code/pipeline validation is on Ada. Real Google Colab
execution is **pending**. The notebook records whether its cells actually ran
in Colab. Package installation/download behavior and other GPU models remain
unvalidated until that run. Select a GPU runtime, then Run all. A missing or
incompatible GPU fails explicitly; CPU fallback is not silently relabeled GPU.


In [ ]:
from pathlib import Path
import os, sys, json, time, subprocess, hashlib, platform, zipfile
STARTED = time.perf_counter()
IS_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_BACKEND_VERSION"))
VALIDATION = os.environ.get("ELISA_NOTEBOOK_VALIDATE") == "1"
CODE_SHA = 'a3a83b6b4ee69a8244f772a03e19a8e1c669d408'
REPO = Path(os.environ.get("ELISA_NOTEBOOK_REPO", "/content/elisa-observer"))
RUN = Path(os.environ.get("ELISA_NOTEBOOK_RUN", "/content/elisa-observer-results"))
RUN.mkdir(parents=True, exist_ok=True)
BATCH = int(os.environ.get("ELISA_NOTEBOOK_BATCH", "1"))  # increase to4 after the first run
REPEATS = 1  # cold+one changed warm batch; increase for supported statistics
FIELD_LEVELS = (0, 1)  # CLI2/3controls are reported separately; keep default for Ada notebook validation
COMMON_LEVEL = max(FIELD_LEVELS)
assert 1 <= BATCH <= 8
# Full setup wall is separate from synthesis/likelihood timings.
SETUP_STARTED = time.perf_counter()


## Pinned code and environment

Public inputs are versioned repository passband, mesh and sparse LD/table test
assets, pinned by the code commit and checked by the scientific source manifest.
No private data or external TESS download is required. The55runtime dependency
pins below come from the tested Ada environment; the CUDA13 stack may require a
newer Colab driver than the runtime provides. Inspect actual versions and stop
on incompatibility. Installation uses an isolated virtual environment and does
not replace Colab's shared Python packages. Setup/network costs are measured
separately and included in total handoff wall time.


In [ ]:
if not VALIDATION:
    if not (REPO / ".git").exists():
        REPO.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "init", str(REPO)], check=True, capture_output=True)
        subprocess.run(["git", "-C", str(REPO), "remote", "add", "origin", "https://github.com/paulnord/elisa.git"], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", CODE_SHA], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
actual_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual_head == CODE_SHA, (actual_head, CODE_SHA)
# The notebook lives in a follow-up commit; pin scientific sources to CODE_SHA.
PINS = {'astropy': '8.0.2', 'astropy-iers-data': '0.2026.11.16.0.37.44', 'attrs': '26.1.0', 'certifi': '2026.10.23', 'charset-normalizer': '3.5.2', 'contourpy': '1.4.0', 'corner': '2.3.0', 'cycler': '0.12.1', 'emcee': '3.1.6', 'fonttools': '4.68.0', 'idna': '3.20', 'jax': '0.12.0.1', 'jax-cuda13-pjrt': '0.12.0.1', 'jax-cuda13-plugin': '0.12.0.1', 'jaxlib': '0.12.0.1', 'jsonschema': '4.26.0', 'jsonschema-specifications': '2025.9.1', 'kiwisolver': '1.5.1', 'llvmlite': '0.50.0', 'matplotlib': '3.11.2', 'ml-dtypes': '0.7.0', 'numba': '0.68.1', 'numpy': '2.5.5', 'nvidia-cublas': '13.10.0.20', 'nvidia-cuda-cccl': '13.3.5.0.6', 'nvidia-cuda-crt': '13.5.37', 'nvidia-cuda-cupti': '13.5.25', 'nvidia-cuda-nvcc': '13.5.37', 'nvidia-cuda-nvrtc': '13.5.37', 'nvidia-cuda-runtime': '13.5.37', 'nvidia-cudnn-cu13': '9.28.1.4', 'nvidia-cufft': '12.5.0.14', 'nvidia-cusolver': '12.4.1.2', 'nvidia-cusparse': '12.9.4.2', 'nvidia-nccl-cu13': '2.32.6', 'nvidia-nvjitlink': '13.5.37', 'nvidia-nvshmem-cu13': '3.8.0', 'nvidia-nvvm': '13.5.37', 'opt-einsum': '3.4.0', 'packaging': '26.3', 'pandas': '3.1.1', 'parameterized': '0.9.0', 'pillow': '13.0.0', 'pyerfa': '2.0.1.5', 'pyparsing': '3.3.3', 'python-dateutil': '4.0.0', 'pyyaml': '6.0.3', 'referencing': '0.37.0', 'requests': '2.34.2', 'rpds-py': '2026.9.1', 'scipy': '1.18.1', 'tqdm': '4.70.1', 'typing-extensions': '4.16.0', 'tzdata': '2026.6', 'urllib3': '2.8.0'}
requirements = "".join(f"{name}=={version}\n" for name,version in PINS.items())
(RUN / "requirements.txt").write_text(requirements)
if VALIDATION:
    PYTHON = Path(sys.executable)
else:
    venv = RUN / "venv"
    subprocess.run([sys.executable, "-m", "venv", str(venv)], check=True)
    PYTHON = venv / "bin/python"
    subprocess.run([str(PYTHON), "-m", "pip", "install", "-r", str(RUN / "requirements.txt")], check=True)
# Source import avoids unpinned build dependencies; commit+manifest pin Elisa.
ENV = dict(os.environ)
for key in ["LD_LIBRARY_PATH", "JAX_PLATFORMS", "JAX_PLATFORM_NAME"]:
    ENV.pop(key, None)
ENV.update(PYTHONPATH=str(REPO / "src")+os.pathsep+str(REPO), JAX_ENABLE_X64="True",
           XLA_PYTHON_CLIENT_PREALLOCATE="false", PYTHONNOUSERSITE="1",
           OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", NUMEXPR_NUM_THREADS="1")
config = RUN / "config.ini"
config.write_text("[general]\nhome = "+str(RUN / "elisa-home")+
    "\n[computational]\nmesh_generator = trapezoidal\nmax_discretization_factor = 12\nnumber_of_processes = 1\nnumber_of_threads = 1\n"+
    "[support]\nld_tables = "+str(REPO / "tests/data/light_curves/limbdarkening")+
    "\ncastelli_kurucz_04_atm_tables = "+str(REPO / "tests/data/ck04")+
    "\nkurucz_93_atm_tables = "+str(RUN / "unused-k93")+"\n")
ENV["ELISA_CONFIG"] = str(config)
SETUP_SECONDS = time.perf_counter()-SETUP_STARTED
print(json.dumps({"code_commit":CODE_SHA,"setup_seconds":SETUP_SECONDS,
    "actual_colab":IS_COLAB,"validation_mode":VALIDATION,"batch":BATCH,"warm_repeats":REPEATS}, indent=2))


## Actual runtime and FP64 gate

All three backends use the same CPU affinity and one-thread BLAS/OMP/MKL/NumExpr
environment. JAX/XLA thread-pool size is not assumed to equal those environment
counts. This is a serial original-CPU comparison, not a tuned multiprocess CPU
benchmark. GPU/runtime availability and FP64 capability are checked explicitly.


In [ ]:
manifest = json.loads((REPO / "docs/ada_validation/background-m019-source-hashes.json").read_text())
assert all(hashlib.sha256((REPO/name).read_bytes()).hexdigest()==digest for name,digest in manifest.items())
PROBE = 'import json,sys,platform,os,jax,numpy as np\nfrom importlib.metadata import version\nexpected=json.loads(sys.argv[1]);actual={n:version(n) for n in expected}\nassert actual==expected, {n:(expected[n],actual[n]) for n in expected if expected[n]!=actual[n]}\nassert jax.config.x64_enabled\ntry: gpu=jax.devices("gpu")[0]\nexcept Exception as exc: raise RuntimeError("No compatible CUDA13 JAX GPU; stop this handoff on this runtime") from exc\na=jax.device_put(np.array([1.,2.,3.],dtype=np.float64),gpu);out=(a*a).block_until_ready()\nassert out.dtype==np.float64 and out.device.platform=="gpu"\nprint(json.dumps({"python":sys.version.split()[0],"machine":platform.machine(),"GPU":gpu.device_kind,"x64":True,\n "affinity":sorted(os.sched_getaffinity(0)) if hasattr(os,"sched_getaffinity") else None,"dependencies":actual}))\n'
RUNTIME = json.loads(subprocess.check_output([str(PYTHON), "-c", PROBE, json.dumps(PINS)], cwd=REPO, env=ENV, text=True).splitlines()[-1])
try:
    RUNTIME["nvidia_smi"] = subprocess.check_output(["nvidia-smi", "--query-gpu=name,driver_version,memory.total,pcie.link.gen.current,pcie.link.width.current", "--format=csv,noheader"],text=True).strip()
except FileNotFoundError:
    RUNTIME["nvidia_smi"] = "unavailable"
print(json.dumps(RUNTIME, indent=2))


In [ ]:
# Read-only CPU identification; no numerical settings or model inputs change.
cpu_path = Path("/proc/cpuinfo")
model_rows = cpu_path.read_text().splitlines() if cpu_path.exists() else []
RUNTIME["CPU_model"] = next((row.split(":",1)[1].strip() for row in model_rows if row.startswith("model name")), platform.processor() or "unknown")
RUNTIME["logical_CPU_count"] = os.cpu_count()
RUNTIME["link_snapshot_scope"] = "Before benchmarks; idle PCIe values do not establish active link capability."
print(json.dumps({key:RUNTIME[key] for key in ["CPU_model","logical_CPU_count","affinity","link_snapshot_scope"]},indent=2))


## Complete matched comparisons

Each command includes fresh physical serialization, candidate construction,
two finite field curves, all CPU bridges, transfers/synchronization, host curve
outputs, original phase/l3/mean/noise/prior/density and numerical accuracy checks.
Fixed setup, first encountered compilation and subsequent changed warm calls
are separated. The overall timer includes imports/process startup. Increase
BATCH/REPEATS only after this run succeeds; one warm sample is not a stable
throughput/tail estimate. No speedup assertion is encoded.


In [ ]:
PROFILES = {}
COMMAND_SECONDS = {}
for backend in ["reference", "cpu", "gpu"]:
    selected = dict(ENV)
    if backend != "gpu":
        selected["JAX_PLATFORMS"] = "cpu"
    output = RUN / f"complete-{backend}.json"
    command = [str(PYTHON), "scripts/validate_gpu_observer.py", "--backend", backend,
               "--batch", str(BATCH), "--repeats", str(REPEATS), "--levels", *map(str, FIELD_LEVELS),
               "--integration-level", str(COMMON_LEVEL), "--output", str(output)]
    if backend == "gpu":
        command.append("--verified-bounded-gpu-window")  # on Ada, outer registered unit enforces this
    t = time.perf_counter()
    with (RUN / f"{backend}.log").open("w") as log:
        result = subprocess.run(command, cwd=REPO, env=selected, stdout=log, stderr=subprocess.STDOUT)
    COMMAND_SECONDS[backend] = time.perf_counter()-t
    if result.returncode:
        raise RuntimeError(f"{backend} command failed; inspect its local log. No valid speedup result.")
    PROFILES[backend] = json.loads(output.read_text())
    print(backend, [round(s["enclosing_complete_batch_seconds"],4) for s in PROFILES[backend]["samples"]])


## Parity, finite-error stop and downloadable evidence

The unchanged raw/component3e-13 relative and normalized3e-13/2e-12 gates apply;
posteriors retain the1e-6 absolute gate. Compare physical identities and adaptive
counts. Packet byte hashes may differ under device arithmetic; that does not
replace flux/density gates. The original1e-4 finite-field error stop remains
independent of nuisance-noise inflation. A parity pass is not a fit qualification.


In [ ]:
# Persist the portable analysis code, then execute with the pinned Python.
ANALYSIS = 'import numpy as np\nreference=PROFILES["reference"]\nparity={}\nfor backend in ["cpu","gpu"]:\n    max_curve=max_posterior=0.\n    for a,b in zip(reference["samples"],PROFILES[backend]["samples"],strict=True):\n        for r,s in zip(a["rows"],b["rows"],strict=True):\n            assert r["physical"]==s["physical"] and r["phase_records"]==s["phase_records"]\n            for model in r["curves"]:\n                for attr in ["fluxes","raw_fluxes","component_fluxes"]:\n                    for band in r["curves"][model][attr]:\n                        x,y=np.asarray(s["curves"][model][attr][band]),np.asarray(r["curves"][model][attr][band])\n                        np.testing.assert_allclose(x,y,rtol=3e-13,atol=2e-12 if attr=="fluxes" else 0)\n                        if attr=="fluxes":max_curve=max(max_curve,float(np.max(abs(x-y))))\n            for band in r["prediction"]:\n                np.testing.assert_allclose(s["prediction"][band],r["prediction"][band],rtol=3e-13,atol=2e-12)\n            np.testing.assert_allclose(s["posterior"],r["posterior"],rtol=0,atol=1e-6)\n            max_posterior=max(max_posterior,abs(s["posterior"]-r["posterior"]))\n            assert s["accuracy_abort"]==r["accuracy_abort"]\n    parity[backend]={"max_normalized_curve_absolute":max_curve,"max_posterior_absolute":max_posterior}\ncosts={d:{"fixed_setup_seconds":q["fixed_input_setup_seconds"],"engine_constructor_seconds":q["constructor_seconds"],\n          "cold_complete_batch_seconds":q["samples"][0]["enclosing_complete_batch_seconds"],\n          "warm_changed_complete_seconds":[s["enclosing_complete_batch_seconds"] for s in q["samples"][1:]],\n          "command_seconds":COMMAND_SECONDS[d]} for d,q in PROFILES.items()}\nmedian=lambda a:float(np.median(a))\nspeedup=median(costs["reference"]["warm_changed_complete_seconds"])/median(costs["gpu"]["warm_changed_complete_seconds"])\nprojections={}\nfor count in [1,10,100]:\n    projected={d:q["fixed_setup_seconds"]+q["engine_constructor_seconds"]+q["cold_complete_batch_seconds"]+\n                  (count-1)*median(q["warm_changed_complete_seconds"]) for d,q in costs.items()}\n    projections[str(count)]={"projected_seconds":projected,"CPU_over_GPU":projected["reference"]/projected["gpu"]}\nqualified=all(not row["accuracy_abort"] for q in PROFILES.values() for s in q["samples"] for row in s["rows"])\nSUMMARY={"code_commit":CODE_SHA,"actual_colab":IS_COLAB,"validation_mode":VALIDATION,\n "colab_status":"executed_on_this_runtime" if IS_COLAB else "pending_real_Colab_execution",\n "runtime":RUNTIME,"source_manifest_sha256":hashlib.sha256((REPO/"docs/ada_validation/background-m019-source-hashes.json").read_bytes()).hexdigest(),\n "batch":BATCH,"warm_repeats":REPEATS,"parity":parity,"costs":costs,"observed_warm_CPU_over_GPU_ratio":speedup,\n "strict_finite_pair_gate_passed":qualified,"continuum_certified":False,"fit_or_MCMC_qualified":False,\n "conditional_constant_warm_cost_reuse_projection":projections,\n "projection_limit":"Uses few observed warm samples as constant future costs; not a measured analysis/sampler throughput or speedup prediction.",\n "network_installation_executed":not VALIDATION,\n "setup_seconds":SETUP_SECONDS,"total_handoff_seconds":time.perf_counter()-STARTED,\n "acceptance":"STOP: finite accuracy controls failed; no fit/sampler acceptance" if not qualified else "Finite pair only; continuum/model adequacy and fitting/MCMC remain unqualified"}\n(RUN/"summary.json").write_text(json.dumps(SUMMARY,indent=2)+chr(10))\nprint(json.dumps(SUMMARY,indent=2))\nARCHIVE=RUN/"elisa-observer-evidence.zip"\nwith zipfile.ZipFile(ARCHIVE,"w",zipfile.ZIP_DEFLATED) as archive:\n    for path in [RUN/"summary.json",RUN/"requirements.txt",*[RUN/f"complete-{d}.json" for d in PROFILES]]:\n        archive.write(path,arcname=path.name)\n'
payload={"PROFILES":PROFILES,"COMMAND_SECONDS":COMMAND_SECONDS,"CODE_SHA":CODE_SHA,"IS_COLAB":IS_COLAB,
         "VALIDATION":VALIDATION,"RUNTIME":RUNTIME,"BATCH":BATCH,"REPEATS":REPEATS,"SETUP_SECONDS":SETUP_SECONDS,
         "STARTED":STARTED,"RUN":str(RUN),"REPO":str(REPO)}
(RUN/"analysis-input.json").write_text(json.dumps(payload))
bootstrap = "from pathlib import Path; import json,time,zipfile,hashlib; p=json.loads(Path("+repr(str(RUN/"analysis-input.json"))+").read_text());globals().update(p);RUN=Path(RUN);REPO=Path(REPO);exec("+repr(ANALYSIS)+")"
# Colab download runs in host below; scientific analysis itself uses pinned dependencies.
bootstrap=bootstrap.replace("if IS_COLAB:\n    from google.colab import files\n    files.download(str(ARCHIVE))", "")
subprocess.run([str(PYTHON),"-c",bootstrap],cwd=REPO,env=ENV,check=True)
SUMMARY=json.loads((RUN/"summary.json").read_text())
if IS_COLAB:
    from google.colab import files
    files.download(str(RUN/"elisa-observer-evidence.zip"))


A finite-error stop is a scientific result, not a failed parity test or a
physical-prior penalty. Keep the summary and accuracy evidence when sharing
results. Report the actual observed CPU/GPU ratio, runtime and output contract;
do not label a stage ratio, compile-excluded lower bound or short diagnostic
batch as converged MCMC or a complete scientific analysis speedup.
